# protoype for the four lists algorithm on the m13 groupoid


In [1]:
import numpy as np
 
# define the permutation class
def tuple_order(t1, t2):
    for i in range(len(t1)):
        if t1[i]< t2[i]:
            return True
        elif t1[i]> t2[i]:
            return False
    return False

class permutation:
    def __init__(self, arr):
        for i in range(len(arr)):
            if i not in arr:
                raise Exception('Inputs needs to be an array of the integers 0 through to N')
        self.arr = tuple(arr)
        self.len = len(arr)
        
    def inv(self):
        return permutation([self.arr.index(i) for i in range(self.len)])

    def __repr__(self):
        return f"permutation({self.arr})"

    def __mul__(self, other):
        """Handles operations like vector * scalar or vector * vector"""
        if isinstance(other, permutation) and self.len == other.len:
            # Scalar multiplication
            return permutation([other.arr[self.arr[i]] for i in range(self.len)])
        elif isinstance(other, int):
            pass
        elif self.len != other.len:
            raise ValueError('input permutations must be of the same length')
        else:
            return NotImplemented # Signal Python to try __rmul__ on the other operand's type
        
    def __eq__(self, other):
        if not isinstance(other, permutation):
            return NotImplemented
        return self.arr == other.arr
    
    def __hash__(self):
        # Combine hashable attributes into a tuple and hash the tuple
        return hash((self.arr, self.len))
    
    def __lt__(self, other):
        if not isinstance(other, permutation):
            return NotImplemented
        # Define the primary sorting order: age, then grade, then name
        return tuple_order(self.arr, other.arr)
    
    def __le__(self, other):
        """
        Defines the behavior for the less than or equal to (<=) operator.
        Compares students based on their 'score' attribute.
        """
        if isinstance(other, permutation):
            return tuple_order(self.arr, other.arr) or self.arr == other.arr
        else:
            # Optional: handle cases where 'other' is not a Student instance
            # by raising an error or returning NotImplemented.
            return NotImplemented

def identity(n):
    X = list(range(n))
    return permutation(X)

def create_supergenerator(generators, n = 13, t = 4):
    """
    Docstring for create_supergenerators
    
    :param generators: Description
    :param n: Length of the word we are searching for 
    :param t: number of supergenerators we search over
    """
    # compose n/t generators from our generator set to obtain L1
    # dictionary of words from the generator words in the generator set indexed by permutations
    L = {}
    q = max(generators[0].arr) # number of elements we are permuting
    e = permutation(list(range(q+1))) # creates the identity permutation. 
    
    L[e] = []
    nt = n//t 
    e*generators[1]
    perms_to_check = []
    perms_to_check.append(e)
    checked_perms = [e]

    for i in range(len(generators)):
        # appends the inverses of the generators 
        generators.append(generators[i].inv())

    for j in range(nt):# goes through the words of length up to nt 
        new_perms = []
        for p in perms_to_check:
            for i in range(len(generators)):
                new_perm = generators[i]*p
                if new_perm in checked_perms:
                    pass
                else:
                    L[new_perm] = L[p] + [i]
                    checked_perms.append(new_perm)
                    new_perms.append(new_perm)
        perms_to_check = new_perms
    return dict(sorted(L.items(), reverse=False)), generators



generators = [permutation([11,0,1,2,3,4,5,6,7,8,9,10]),
              permutation([9,4,6,7,1,8,2,3,5,0,11,10]),
              permutation([0,9,3,2,8,6,5,7,4,1,10,11])]

L, generators = create_supergenerator(generators)
g = permutation([5,2,8,3,0,1,4,7,6,9,10,11])
g = g

# Very important !!
we compose permutations from left to right 
$$
a*b*c 
$$
means we perform a, and then b, and then c in that order

In [2]:
# define the permutation tree class


class tree_node:
    def __init__(self, node_data = None, depth = None):
        if depth is None:
            self.depth = 0
        else:
            self.depth = 1
        self.data = node_data
        self.descendents = {} # empty dictionary for the descendents 

    def __repr__(self):
        return f"node({self.data})"
    
    def subtree(self, path):
        new_tree = self
        for i in path:
            new_tree = new_tree.descendents[i]
        return new_tree

    def add_node(self, node_data, path):
        current_node = self
        depth_count = 0
        for i in path:
            depth_count += 1
            if i in current_node.descendents:
                current_node = current_node.descendents[i]
            else:
                new_node = tree_node(depth = depth_count)
                current_node.descendents[i] = new_node
                current_node = current_node.descendents[i]

        current_node.data = node_data
        return 
    
    def maximum_path(self, perm = None, n = None):
        """
        finds the path on the tree of maximal order in which the descendants of 
        each internal node in the tree are ordered by inv(perm). That is perm.inv.arr[0] is 
        considered to be the new smalles5t entry and perm.inv.arr[-1] is the largest 
        """  
        if perm is None:
            perm = identity(n) 
        new_order  = perm.inv().arr
        current_tree = self
        path = []
        stopped = False
        while not stopped:
            if len(list(current_tree.descendents.keys())) == 0:
                # if there are no descendents then we have hit a leaf node and can stop
                stopped = True 
                break

            # find the highest valued descendent in the current internal node
            found_index = False
            max_index = -1
            while not found_index:
                if new_order[max_index] in current_tree.descendents.keys():
                    found_index = True
                    break
                    
                max_index = max_index - 1 
                if max_index < -1* len(new_order):
                    raise Exception ('none of the edge labels match the values 0 to n')
            path.append(new_order[max_index])
            # moves on to the next layer down 
            current_tree = current_tree.descendents[new_order[max_index]]
        node_data = current_tree.data 
        return path, node_data 

    def next_element(self, perm, path):
        if perm is None:
            perm = identity(n) 
        new_order  = perm.inv().arr
        current_tree = self.subtree(path[:-1])
        next_element_exists = False
        path_index = -1
        for i in range(len(path)):
            # iterate through the path elements in reverse order until we can find an element that can be increased
            path_element = path[path_index]
            perm_index = new_order.index(path_element) # find the path element in the newq order
            higher_valued_edges = new_order[perm_index + 1:] # get everything of a higher value
            available_edges = current_tree.descendents.keys()
            print('loking at the nth element from the right', path_index, 'the element is ', path_element)
            print('higher ordered edges', higher_valued_edges)
            print('available eges', available_edges)
            print(' ')

            # checks whether any elements of the 
            for j in higher_valued_edges:
                if j in available_edges:
                    next_element_exists = True
                    break 
            if next_element_exists:
                break
            else:
                path_index = path_index - 1
                current_tree = self.subtree(path[:path_index])
        if not next_element_exists:
            return 

        # so we know that there is a 
        # append j to the current path and then get the maximal path w.r.t the permutation perm
        next_path_part1 = path[:path_index]
        next_path_part2 = [j]
        remaining_tree_path = next_path_part1 + next_path_part2
        print(remaining_tree_path)
        remaining_tree = self.subtree(remaining_tree_path)
        a,b = remaining_tree.maximum_path(perm = perm )
        next_path_part3 = a 
        print('the three components of the path')
        print(next_path_part1)
        print(next_path_part2)
        print(next_path_part3)
        next_path = next_path_part1 + next_path_part2 + next_path_part3
        if len(next_path) != len(path):
            print(' paths must be of the same length somethingn has gone wronmg if you can read this')
        next_perm = self.subtree(next_path).data
        print('next path', next_path)
        print('next path', next_perm)
        return next_path, next_perm
    

def next_seq(seq, base):
    index = -1
    while seq[index] == base -1:
        seq[index] = 0
        index -= 1
    seq[index] += 1
    return seq

def create_tree(L):
    """
    creates a tree from the entries of a list of permutations L
    The leaf nodes of the tree are sorted in lexicographic order. 
    """
    # the tree is stored as a dictionary of dictionaries. 
    permutation_list = list(L.keys())
    
    # our permutation can be a base q number 
    base = max(permutation_list[0].arr) + 1

    # number opf digits in the representation of the paths 
    n = int(np.ceil(np.log(len(permutation_list))/np.log(base)))

    # every permutation can be indexed by a sequence 
    path = n*[0]

    tree = tree_node()
    
    for perm in L:
        tree.add_node(perm, path)
        path = next_seq(path, base)

    return tree



def greater_than(path1, path2, perm):
    n = len(path1)
    if len(path1) != len(path2):
        raise Exception('paths must be of equal length')
    perm_array = perm.arr
    index = n
    for i in range(len(path1)):
        index = index -1
        value_1 = index 
    
    pass



generators = [permutation([11,0,1,2,3,4,5,6,7,8,9,10]),
              permutation([9,4,6,7,1,8,2,3,5,0,11,10]),
              permutation([0,9,3,2,8,6,5,7,4,1,10,11])]

L, generators = create_supergenerator(generators)
tree = create_tree(L)



In [25]:
# Now for the M13 bit 

# you need to create a list of 13 trees each one 
# lines in the projective plane M13
lines = []
lines.append((1,6,8,10))
lines.append((1,2,3,0))
lines.append((1,5,9, 12))
lines.append((1,4,7,11))
lines.append((2,5,8,11))
lines.append((3,5,7,10))
lines.append((3,6,9,11))
lines.append((4,5,6,0))
lines.append((7,8,9,0))
lines.append((9,2,4,10))
lines.append((10,11,12,0))
lines.append((3,4,8,12))
lines.append((7,2,6,12))


generator_dictionary = {}
for i in range(13):
    generator_abbreviation = {}
    generator_dictionary[i] = []
    # creates a dictionary of the allowable permutations where the empty vertex is at node i
    for line in lines:
        if i in line:
            # define the relevant permutation and add them to the dictionary 
            for k in line:
                a = [0,1,2,3,4,5,6,7,8,9,10,11,12] # initial permutations
                if k == i:
                    pass 
                else:
                    remaining_elements = [x for x in line if x not in [i, k]]
                    a[i] = k 
                    a[k] = i
                    a[remaining_elements[0]] = remaining_elements[1]
                    a[remaining_elements[1]] = remaining_elements[0]
                    generator_dictionary[i].append(permutation(a))



def list_permutations(initial_empty_vertex = 0, depth = 3):
    # lists all the permutations that can be acheived in the M13 groupoid starting at teh origin
    # every permutation is abbreviated using a pair of numbers (a,b) where a represents the location of the 
    # empty vertex at the start of the permutation and b represents the location to which the empty vertex is mapped
    perm_dictionary = {}
    perm_dictionary[identity(13)] = []
    for i in range(depth):
        # go through the current vertices we have in the list
        new_elements = []
        for perm in perm_dictionary.keys():
            # find the location of zero
            empty_node_location = perm.arr.index(initial_empty_vertex)
            next_step_generators = generator_dictionary[empty_node_location]
            # take the relevant generators and multiply them by
            
            for generator in next_step_generators:
                new_permutation = perm * generator
                new_empty_node_location = generator.arr.index(empty_node_location)
                new_path = perm_dictionary[perm] + [(empty_node_location, new_empty_node_location)]
                if new_permutation in perm_dictionary:
                    pass
                else:
                    if len(new_path) > 2:
                        pass
                        # print statements for debugging purposes. unccomment thewm when things go wrong
                        #print((new_permutation, new_path))
                    new_elements.append((new_permutation, new_path))
        for i in new_elements:
            perm_dictionary[i[0]] = i[1]
    return perm_dictionary

def make_tree(depth = 3):
    tree = {}
    path_dictionary = {}
    for i in range(13):
        tree[i] = []
        for _ in range(depth + 1):
            tree[i].append([])
        path_dictionary_i = list_permutations(initial_empty_vertex = i, depth = depth)
        path_dictionary[i] = path_dictionary_i
        for perm in path_dictionary_i:
            path = path_dictionary_i[perm]
            path_length = len(path)
            tree[i][path_length].append(perm)
    return tree, path_dictionary

A = list_permutations(initial_empty_vertex=3)

In [26]:
a = []
a.append([])
a

[[]]

In [27]:
tree, paths = make_tree(depth = 4)

In [28]:
tree[0][1]

[permutation((1, 0, 3, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12)),
 permutation((2, 3, 0, 1, 4, 5, 6, 7, 8, 9, 10, 11, 12)),
 permutation((3, 2, 1, 0, 4, 5, 6, 7, 8, 9, 10, 11, 12)),
 permutation((4, 1, 2, 3, 0, 6, 5, 7, 8, 9, 10, 11, 12)),
 permutation((5, 1, 2, 3, 6, 0, 4, 7, 8, 9, 10, 11, 12)),
 permutation((6, 1, 2, 3, 5, 4, 0, 7, 8, 9, 10, 11, 12)),
 permutation((7, 1, 2, 3, 4, 5, 6, 0, 9, 8, 10, 11, 12)),
 permutation((8, 1, 2, 3, 4, 5, 6, 9, 0, 7, 10, 11, 12)),
 permutation((9, 1, 2, 3, 4, 5, 6, 8, 7, 0, 10, 11, 12)),
 permutation((10, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 12, 11)),
 permutation((11, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 0, 10)),
 permutation((12, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 10, 0))]

In [29]:
tree[8][1]

[permutation((0, 8, 2, 3, 4, 5, 10, 7, 1, 9, 6, 11, 12)),
 permutation((0, 10, 2, 3, 4, 5, 8, 7, 6, 9, 1, 11, 12)),
 permutation((0, 6, 2, 3, 4, 5, 1, 7, 10, 9, 8, 11, 12)),
 permutation((0, 1, 8, 3, 4, 11, 6, 7, 2, 9, 10, 5, 12)),
 permutation((0, 1, 11, 3, 4, 8, 6, 7, 5, 9, 10, 2, 12)),
 permutation((0, 1, 5, 3, 4, 2, 6, 7, 11, 9, 10, 8, 12)),
 permutation((9, 1, 2, 3, 4, 5, 6, 8, 7, 0, 10, 11, 12)),
 permutation((7, 1, 2, 3, 4, 5, 6, 0, 9, 8, 10, 11, 12)),
 permutation((8, 1, 2, 3, 4, 5, 6, 9, 0, 7, 10, 11, 12)),
 permutation((0, 1, 2, 8, 12, 5, 6, 7, 3, 9, 10, 11, 4)),
 permutation((0, 1, 2, 12, 8, 5, 6, 7, 4, 9, 10, 11, 3)),
 permutation((0, 1, 2, 4, 3, 5, 6, 7, 12, 9, 10, 11, 8))]

In [30]:
tree[0][3][2]

permutation((6, 1, 2, 3, 4, 5, 0, 7, 10, 9, 8, 11, 12))

In [43]:
import copy

def index_pair_lists(n):
  p = [1,0]
  p_list = [[1,0]]
  while p[1] < n:
    if p[0] > p[1]:
      p[1] += 1
    else:
      p[0] += 1
    p_list.append(copy.deepcopy(p))
  return p_list

index_pair_lists(4)

def compare_permutation_lists(list_0, list_1, target_permutation):
  """
  Algorithm that goes through two lists of permutations
  sorts the permutations
  iterates through the lists comparing the miinum elements
  """
  list_1_copy = []
  for i in range(len(list_1)):
    perm_i = list_1[i]
    list_1_copy.append(target_permutation*perm_i)
  # sort list 1
  list_1_copy.sort()
  list_0_copy = copy.deepcopy(list_0)
  list_0_copy.sort()

  

  while len(list_0_copy) > 0 and len(list_1_copy) > 0:
    print('list 0 ', list_0_copy)
    print('list 1 ', list_1_copy)
    minimum_perm_0 = list_0_copy[0]
    minimum_perm_1 = list_1_copy[0]
    if minimum_perm_0 == minimum_perm_1:
      return True, minimum_perm_0, minimum_perm_1
    elif minimum_perm_0 < minimum_perm_1:
      list_0_copy.pop(0)
    elif minimum_perm_1 < minimum_perm_0:
      list_1_copy.pop(0)
  return False, minimum_perm_0, minimum_perm_1

"""
This function assumes that there is a global variable called tree\
tree is a dictionary whose keys are the natural numbers 0,1,2,3,..,12
These represent the vertices in PG(2,3)
"""


def two_list(target_perm, path_dictionary, tree, n = 3):
  vertex_0 = 0 # vertex 0 represents starting at the origin
  vertex_1 = target_perm.inv().arr[0] # the location of the empty vertex in the ptarget permutation
  print('vertex 1', vertex_1)
  index_pairs = index_pair_lists(n)
  for index_pair in index_pairs:
    print('updating index pair')
    print(' ')
    print('index pair', index_pair)
    i_0 = index_pair[0]
    i_1 = index_pair[1]

    # select the relevant list of permutations
    perm_list_0 = tree[vertex_0][i_0]
    perm_list_1 = tree[vertex_1][i_1]

    flag, p_0, p_1 = compare_permutation_lists(perm_list_0,
                                              perm_list_1,
                                              target_perm)
    if flag:
      # return the permutations and their generators.
      generators_0 = path_dictionary[vertex_0][p_0]
      generators_1 = path_dictionary[vertex_1][p_1]
      return (p_0, generators_0), (p_1, generators_1)
  return False

tree, path_dictionary = make_tree(depth = 3)


tree, path_dictionary = make_tree(depth = 3)
target_perm = permutation([8, 0, 3, 2, 4, 5, 10, 7, 1, 9, 6, 11, 12])
two_list(target_perm, path_dictionary, tree, n = 3)

vertex 1 1
updating index pair
 
index pair [1, 0]
list 0  [permutation((1, 0, 3, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12)), permutation((2, 3, 0, 1, 4, 5, 6, 7, 8, 9, 10, 11, 12)), permutation((3, 2, 1, 0, 4, 5, 6, 7, 8, 9, 10, 11, 12)), permutation((4, 1, 2, 3, 0, 6, 5, 7, 8, 9, 10, 11, 12)), permutation((5, 1, 2, 3, 6, 0, 4, 7, 8, 9, 10, 11, 12)), permutation((6, 1, 2, 3, 5, 4, 0, 7, 8, 9, 10, 11, 12)), permutation((7, 1, 2, 3, 4, 5, 6, 0, 9, 8, 10, 11, 12)), permutation((8, 1, 2, 3, 4, 5, 6, 9, 0, 7, 10, 11, 12)), permutation((9, 1, 2, 3, 4, 5, 6, 8, 7, 0, 10, 11, 12)), permutation((10, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 12, 11)), permutation((11, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 0, 10)), permutation((12, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 10, 0))]
list 1  [permutation((8, 0, 3, 2, 4, 5, 10, 7, 1, 9, 6, 11, 12))]
list 0  [permutation((2, 3, 0, 1, 4, 5, 6, 7, 8, 9, 10, 11, 12)), permutation((3, 2, 1, 0, 4, 5, 6, 7, 8, 9, 10, 11, 12)), permutation((4, 1, 2, 3, 0, 6, 5, 7, 8, 9, 10, 11, 12)), permutatio

((permutation((1, 0, 3, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12)), [(0, 1)]),
 (permutation((1, 0, 3, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12)), [(1, 0)]))

In [ ]:
path_dictionary[0]


{permutation((0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12)): [],
 permutation((1, 0, 3, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12)): [(0, 1)],
 permutation((2, 3, 0, 1, 4, 5, 6, 7, 8, 9, 10, 11, 12)): [(0, 2)],
 permutation((3, 2, 1, 0, 4, 5, 6, 7, 8, 9, 10, 11, 12)): [(0, 3)],
 permutation((4, 1, 2, 3, 0, 6, 5, 7, 8, 9, 10, 11, 12)): [(0, 4)],
 permutation((5, 1, 2, 3, 6, 0, 4, 7, 8, 9, 10, 11, 12)): [(0, 5)],
 permutation((6, 1, 2, 3, 5, 4, 0, 7, 8, 9, 10, 11, 12)): [(0, 6)],
 permutation((7, 1, 2, 3, 4, 5, 6, 0, 9, 8, 10, 11, 12)): [(0, 7)],
 permutation((8, 1, 2, 3, 4, 5, 6, 9, 0, 7, 10, 11, 12)): [(0, 8)],
 permutation((9, 1, 2, 3, 4, 5, 6, 8, 7, 0, 10, 11, 12)): [(0, 9)],
 permutation((10, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 12, 11)): [(0, 10)],
 permutation((11, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 0, 10)): [(0, 11)],
 permutation((12, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 10, 0)): [(0, 12)],
 permutation((6, 0, 3, 2, 4, 5, 1, 7, 10, 9, 8, 11, 12)): [(0, 1), (1, 6)],
 permutation((8, 0, 3, 2, 4, 5, 10, 7, 1, 9

In [9]:
b = target_perm.inv()
print(b*target_perm)

permutation((0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12))


In [12]:
target_perm.inv().arr[0]

8

In [14]:
generator_dictionary

{0: [permutation((1, 0, 3, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12)),
  permutation((2, 3, 0, 1, 4, 5, 6, 7, 8, 9, 10, 11, 12)),
  permutation((3, 2, 1, 0, 4, 5, 6, 7, 8, 9, 10, 11, 12)),
  permutation((4, 1, 2, 3, 0, 6, 5, 7, 8, 9, 10, 11, 12)),
  permutation((5, 1, 2, 3, 6, 0, 4, 7, 8, 9, 10, 11, 12)),
  permutation((6, 1, 2, 3, 5, 4, 0, 7, 8, 9, 10, 11, 12)),
  permutation((7, 1, 2, 3, 4, 5, 6, 0, 9, 8, 10, 11, 12)),
  permutation((8, 1, 2, 3, 4, 5, 6, 9, 0, 7, 10, 11, 12)),
  permutation((9, 1, 2, 3, 4, 5, 6, 8, 7, 0, 10, 11, 12)),
  permutation((10, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 12, 11)),
  permutation((11, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 0, 10)),
  permutation((12, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 10, 0))],
 1: [permutation((0, 6, 2, 3, 4, 5, 1, 7, 10, 9, 8, 11, 12)),
  permutation((0, 8, 2, 3, 4, 5, 10, 7, 1, 9, 6, 11, 12)),
  permutation((0, 10, 2, 3, 4, 5, 8, 7, 6, 9, 1, 11, 12)),
  permutation((3, 2, 1, 0, 4, 5, 6, 7, 8, 9, 10, 11, 12)),
  permutation((2, 3, 0, 1, 4, 5, 6, 7, 8, 9, 10, 

In [36]:
tree.next_element(identity(12), [4,3])

tree2 = tree.subtree([4,4])
tree2.maximum_path(n = 12)

loking at the nth element from the right -1 the element is  3
higher ordered edges (4, 5, 6, 7, 8, 9, 10, 11)
available eges dict_keys([0, 1, 2, 3, 4])
 
[4, 4]
the three components of the path
[4]
[4]
[]
next path [4, 4]
next path permutation((11, 10, 9, 4, 6, 7, 1, 8, 2, 3, 5, 0))


([], permutation((11, 10, 9, 4, 6, 7, 1, 8, 2, 3, 5, 0)))

In [41]:
tree.maximum_path(n = 12)

([4, 4], permutation((11, 10, 9, 4, 6, 7, 1, 8, 2, 3, 5, 0)))

In [8]:
a= [1,2,3,4,5,6,7,8,9]
a[:-1]

[1, 2, 3, 4, 5, 6, 7, 8]

In [26]:
root_node = tree_node()
dir(root_node)
root_node.data = 5
root_node.data

root_node.add_node('twenty', [1,2,3])
root_node.descendents[1].depth
root_node.subtree([1,2]).data = 'six'
root_node.descendents[1].descendents[2].descendents[3]

node(twenty)

# What is it that you need to do 
you need to implement the following 
- Class for permutation trees
- Function that maps paths onto permutations
- Function that sorts the paths in a permutation tree based on a given permutation